<a href="https://colab.research.google.com/github/RomanPhys/Avito/blob/main/external_context_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Внешний контекст, версия v5

Ноутбук отвечает на два вопроса.

Первый про рекомендации. Дают ли агрегации из статьи что-то на настоящем SASRec авторов, если выбирать эпоху честно и сравнивать с моделью, которой дали те же лишние эпохи без контекста? И что на состояниях SASRec дают разложение и память переходов? Это часть A, она такая же, как в v4.

Второй вопрос новый и главный. Тезис статьи про контекст, совпадающий по времени, то есть про то, что происходит со всеми клиентами прямо сейчас. Проверять его надо на банковских данных, а не на MovieLens. В части B ноутбук скачивает датасет Churn из Google Drive авторов и на задаче предсказания следующего MCC сравнивает четыре вещи. Это популярность за всё время, текущая популярность (аналог их среднего поля), похожие ситуации за всё время и похожие ситуации только за последние дни. Заодно проверяется критерий нестационарности, который заранее говорит, может ли контекст в целом вообще помочь. Те же сравнения считаются и на состояниях SASRec в части A.

По времени часть A занимает около полутора часов на GPU для трёх разбиений пользователей и ещё минут 45 на процессоре. Часть B занимает около получаса на одно разбиение. Перед запуском включи GPU в меню Runtime, пункт Change runtime type, вариант T4 GPU. Потом запусти все ячейки через Run all.

Если хочется быстрее, можно открыть этот же ноутбук во второй вкладке на обычном CPU и поставить там RUN_SASREC = False. Тогда часть B посчитается параллельно, а итоговая сводка в любой из вкладок соберёт всё, что лежит на Drive.

## Настройки

USE_DRIVE лучше оставить True. Тогда всё сохраняется на Google Drive в папку ext_work_v5, и если Colab оборвёт сессию, достаточно запустить ноутбук заново. Посчитанное он пропустит.

SPLITS задаёт разбиения пользователей для SASRec, BANK_SPLITS для банковских данных. BANK задаёт датасеты авторов, по умолчанию только churn. Можно добавить default, он больше, и для него берётся случайная выборка из BANK_MAX_USERS клиентов.

In [1]:
USE_DRIVE = True
RUN_SASREC = True
SPLITS = [0, 1, 2]
AGGS = "learnable_attention,mean"
EXTRA_EPOCHS = 20
BANK = ["churn"]
BANK_SPLITS = [0]
BANK_MAX_USERS = 12000

import os, json, glob, zipfile, shutil, subprocess, time
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ext_work_v5"
else:
    WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)
print("результаты будут в", WORK)
if RUN_SASREC:
    import torch
    print("GPU есть" if torch.cuda.is_available() else "GPU нет, включи T4 или поставь RUN_SASREC = False")

Mounted at /content/drive
результаты будут в /content/drive/MyDrive/ext_work_v5
GPU есть


## 1. Код

Когда появится кнопка выбора файла, загрузи external_context_v5.zip. Если архив уже лежит в папке ext_work_v5 на Drive, ноутбук возьмёт его оттуда.

In [2]:
CODE = "/content/ext"
if not os.path.exists(f"{CODE}/code/run_time_local.py"):
    shutil.rmtree(CODE, ignore_errors=True)
    found = sorted(glob.glob(f"{WORK}/external_context_v*.zip"))
    if found:
        path = found[-1]
    else:
        from google.colab import files
        path = next(iter(files.upload()))
        if USE_DRIVE:
            shutil.copy(path, WORK)
    zipfile.ZipFile(path).extractall(CODE)
assert os.path.exists(f"{CODE}/code/run_time_local.py"), "это старый архив, нужен external_context_v5.zip"
print("код на месте", sorted(os.listdir(CODE)))


def run(script, W, log, **env):
    # каждый шаг numpy-конвейера в отдельном процессе с чистыми переменными EXT_
    e = {k: v for k, v in os.environ.items() if not k.startswith("EXT_")}
    e.update({k: str(v) for k, v in env.items()})
    e["EXT_WORK"] = W
    t0 = time.time()
    with open(f"{W}/{log}", "w") as f:
        p = subprocess.run(["python", script], env=e, stdout=f, stderr=subprocess.STDOUT, cwd=f"{CODE}/code")
    if p.returncode != 0:
        print(open(f"{W}/{log}").read()[-3000:])
        raise RuntimeError(f"{script} упал, хвост лога выше")
    print(f"    {script} готов за {time.time() - t0:.0f} с")


def show(script, W, *args, **env):
    e = {k: v for k, v in os.environ.items() if not k.startswith("EXT_")}
    e.update({k: str(v) for k, v in env.items()})
    e["EXT_WORK"] = W
    p = subprocess.run(["python", script, *args], env=e, capture_output=True, text=True, cwd=f"{CODE}/code")
    print(p.stdout)
    if p.returncode != 0:
        print(p.stderr[-3000:])

Saving external_context_v5.zip to external_context_v5.zip
код на месте ['README.md', 'code', 'colab', 'results', 'torch_patch']


# Часть A. SASRec авторов на MovieLens-1M

## A1. Репозиторий авторов и данные

В репозитории авторов нет пакета recsys/src/data. Его случайно исключило правило data/ в их .gitignore, поэтому пакет восстанавливается из нашего файла recsys_src_data.py. MovieLens-1M скачивается с сайта GroupLens, в файле с оценками должно быть 1000209 строк.

In [3]:
if RUN_SASREC:
    %cd /content
    !test -d eca || git clone -q --depth 1 https://github.com/petrsokerin/External-Context-Aggregation.git eca
    !mkdir -p eca/recsys/src/data eca/recsys/data/movielens_1m
    !cp {CODE}/torch_patch/recsys_src_data.py eca/recsys/src/data/__init__.py
    !cp {CODE}/torch_patch/sasrec_pipeline.py eca/recsys/
    !test -f /content/ml-1m.zip || wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip -O /content/ml-1m.zip
    !unzip -o -q /content/ml-1m.zip -d /content/
    !cp /content/ml-1m/ratings.dat eca/recsys/data/movielens_1m/
    !wc -l eca/recsys/data/movielens_1m/ratings.dat

/content
1000209 eca/recsys/data/movielens_1m/ratings.dat


## A2. SASRec в коде авторов

Для каждого разбиения скрипт учит SASRec с гиперпараметрами из config/main_movielens_1m.yaml и выбирает эпоху по валидации. Тестовых пользователей авторов он делит на валидацию и тест в пропорции один к двум. Потом выгружает состояния модели, доучивает её 20 эпох без контекста и отдельно по 20 эпох с каждой агрегацией авторов.

В логе сначала идут эпохи обучения, потом строка pretrained с NDCG на val и test, в первом прогоне было 0.121 и 0.110. Дальше идут 20 строк no-context и по 20 строк на каждую агрегацию. Если сессия оборвалась, запусти ячейку ещё раз, готовые разбиения пропускаются.

In [4]:
if RUN_SASREC:
    %cd /content/eca/recsys
    n_aggs = len([a for a in AGGS.split(",") if a])
    for s in SPLITS:
        W = f"{WORK}/split{s}"
        os.makedirs(W, exist_ok=True)
        summ_path = f"{W}/sasrec_summary.json"
        summ = json.load(open(summ_path)) if os.path.exists(summ_path) else {}
        if len(summ.get("authors_agg", {})) >= n_aggs and "extra_epochs_control" in summ:
            print(f"разбиение {s} уже посчитано, пропускаю")
            continue
        skip = "--skip_pretrain" if "pretrained" in summ else ""
        print(f"разбиение {s}", "SASRec уже обучен, беру чекпойнт" if skip else "учу SASRec с нуля")
        !EXT_SPLIT_SEED={s} python sasrec_pipeline.py --config config/main_movielens_1m.yaml --work {W} --seed {s} {skip} --extra_epochs {EXTRA_EPOCHS} --authors_agg {AGGS} --freeze 0 --alpha 0.1 --ext_epochs 20

/content/eca/recsys
разбиение 0 учу SASRec с нуля
interactions 574376, items 3125, users {'train': 4827, 'val': 402, 'test': 805}
True
Epoch 1/150: 100% 151/151 [00:03<00:00, 42.99it/s]
Epoch 2/150: 100% 151/151 [00:02<00:00, 66.29it/s]
Epoch 3/150: 100% 151/151 [00:02<00:00, 66.11it/s]
2
100% 13/13 [00:00<00:00, 52.85it/s]
Epoch 3, Tr Loss: 7.8659, Val Loss 7.6155, Val NDCG@10: 0.0062
Epoch 4/150: 100% 151/151 [00:02<00:00, 55.02it/s]
Epoch 5/150: 100% 151/151 [00:02<00:00, 56.83it/s]
Epoch 6/150: 100% 151/151 [00:02<00:00, 65.66it/s]
5
100% 13/13 [00:00<00:00, 59.74it/s]
Epoch 6, Tr Loss: 7.3502, Val Loss 7.4787, Val NDCG@10: 0.0116
Epoch 7/150: 100% 151/151 [00:02<00:00, 66.14it/s]
Epoch 8/150: 100% 151/151 [00:02<00:00, 64.74it/s]
Epoch 9/150: 100% 151/151 [00:02<00:00, 63.46it/s]
8
100% 13/13 [00:00<00:00, 39.66it/s]
Epoch 9, Tr Loss: 7.2671, Val Loss 7.4679, Val NDCG@10: 0.0126
Epoch 10/150: 100% 151/151 [00:02<00:00, 52.97it/s]
Epoch 11/150: 100% 151/151 [00:02<00:00, 63.20it/s]

## A3. Разложение на состояниях SASRec

Это тот же конвейер на numpy, что в отчёте, только вместо нашего энкодера он берёт выгруженные состояния SASRec. После первого шага ноутбук сверяет базу. NDCG@10 без контекста в numpy должен совпасть с тем, что SASRec показал на тесте в шаге A2. Если не совпадёт, ячейка остановится, и это надо прислать мне.

Последний шаг, run_time_local.py, новый. Он считает схему с похожестью и свежестью и критерий нестационарности. Окна здесь 30, 90 и 365 дней, потому что MovieLens растянут на три года.

In [5]:
if RUN_SASREC:
    for s in SPLITS:
        W = f"{WORK}/split{s}"
        env = dict(EXT_STATES=f"{W}/sasrec_states.npz")
        print(f"разбиение {s}")
        if not os.path.exists(f"{W}/results.json"):
            run("run_context.py", W, "context.log", **env)
            run("run_experiments.py", W, "experiments.stdout", **env)
        base_sasrec = json.load(open(f"{W}/sasrec_summary.json"))["pretrained"]["test"]
        base_numpy = [r for r in json.load(open(f"{W}/results.json"))["A_replication"] if r["name"] == "Without context"][0]["ndcg_last"]
        print(f"    база SASRec {base_sasrec:.4f}, база в numpy {base_numpy:.4f}")
        if abs(base_sasrec - base_numpy) > 0.002:
            raise RuntimeError("базы не совпали, выгрузка состояний неверная, пришли мне этот вывод")
        for script, out, log in (("run_shrink_control.py", "results_shrink_control.json", "shrink.log"),
                                 ("run_diagnostics.py", "diagnostics.json", "diagnostics.log"),
                                 ("run_retrieval.py", "results_retrieval.json", "retrieval.stdout"),
                                 ("run_time_local.py", "results_time_local.json", "time_local.stdout")):
            if not os.path.exists(f"{W}/{out}"):
                run(script, W, log, EXT_WINDOWS="30,90,365", **env)

разбиение 0
    run_context.py готов за 81 с
    run_experiments.py готов за 138 с
    база SASRec 0.1099, база в numpy 0.1099
    run_shrink_control.py готов за 6 с
    run_diagnostics.py готов за 4 с
    run_retrieval.py готов за 387 с
    run_time_local.py готов за 193 с
разбиение 1
    run_context.py готов за 80 с
    run_experiments.py готов за 143 с
    база SASRec 0.0953, база в numpy 0.0953
    run_shrink_control.py готов за 7 с
    run_diagnostics.py готов за 4 с
    run_retrieval.py готов за 399 с
    run_time_local.py готов за 198 с
разбиение 2
    run_context.py готов за 80 с
    run_experiments.py готов за 142 с
    база SASRec 0.0980, база в numpy 0.0980
    run_shrink_control.py готов за 6 с
    run_diagnostics.py готов за 5 с
    run_retrieval.py готов за 394 с
    run_time_local.py готов за 196 с


# Часть B. Банковские данные авторов

## B1. Данные

Ноутбук скачивает файл churn.parquet из папки preprocessed на Google Drive авторов. Ссылка на эту папку есть в README их репозитория. Если скачать автоматически не получится, ячейка скажет об этом. Тогда открой папку https://drive.google.com/drive/folders/1VFGL6ZOtnP7ErPyVlmQv62NJTXUs8ekY, скачай из preprocessed нужный файл, положи его в папку ext_work_v5/bank_data на своём Drive и запусти ячейку снова.

In [6]:
BANK_DIR = f"{WORK}/bank_data"
os.makedirs(BANK_DIR, exist_ok=True)
need = [n for n in BANK if not os.path.exists(f"{BANK_DIR}/{n}.parquet")]
if need:
    !pip -q install -U gdown
    import gdown
    try:
        listing = gdown.download_folder(id="1VFGL6ZOtnP7ErPyVlmQv62NJTXUs8ekY", skip_download=True,
                                        quiet=True, remaining_ok=True)
        for f in listing:
            stem = os.path.basename(f.path).rsplit(".", 1)[0]
            if "preprocessed" in f.path and stem in need and f.path.endswith(".parquet"):
                gdown.download(id=f.id, output=f"{BANK_DIR}/{stem}.parquet", quiet=False)
    except Exception as err:
        print("автоматически скачать не вышло", repr(err)[:300])
for n in BANK:
    p = f"{BANK_DIR}/{n}.parquet"
    if os.path.exists(p):
        import pandas as pd
        df = pd.read_parquet(p, columns=["user_id", "mcc_code", "timestamp"])
        print(n, len(df), "транзакций,", df.user_id.nunique(), "клиентов,", df.mcc_code.nunique(), "MCC")
        print(df.head(3))
    else:
        print(f"нет файла {p}, скачай его вручную по инструкции выше")

автоматически скачать не вышло TypeError("download_folder() got an unexpected keyword argument 'remaining_ok'")
нет файла /content/drive/MyDrive/ext_work_v5/bank_data/churn.parquet, скачай его вручную по инструкции выше


## B2. Расчёт

Для каждого датасета и разбиения клиентов конвейер делает следующее. Учит лёгкий энкодер MS-EMA предсказывать MCC следующей транзакции, собирает причинный контекст и считает агрегации статьи, разложение, контроль g = 0, диагностику и память переходов. В конце идёт новая схема с похожестью и свежестью и критерий нестационарности, окна здесь 7, 30 и 90 дней.

Кеши конвейера лежат рядом с результатами и занимают на Drive в сумме около 3 ГБ. После того как пришлёшь архив, файлы .pkl и .npz из ext_work_v5 можно удалить.

Время в банковских данных известно с точностью до дня, поэтому контекст здесь видит только события строго раньше дня запроса (EXT_STRICT_PAST=1). Иначе в него попадали бы транзакции других клиентов того же дня, которые могли случиться позже.

In [7]:
for n in BANK:
    src = f"{BANK_DIR}/{n}.parquet"
    if not os.path.exists(src):
        print(f"пропускаю {n}, нет данных")
        continue
    for s in BANK_SPLITS:
        W = f"{WORK}/bank_{n}_split{s}"
        os.makedirs(W, exist_ok=True)
        env = dict(EXT_DATASET="bank", EXT_BANK_FILE=src, EXT_BANK_MAX_USERS=BANK_MAX_USERS,
                   EXT_SPLIT_SEED=s, EXT_STRICT_PAST=1, EXT_WINDOWS="7,30,90")
        print(f"{n}, разбиение {s}")
        steps = (("run_encoder2.py", "encoder_msema.npz", "encoder_train.log"),
                 ("run_context.py", "cache_context.pkl", "context.log"),
                 ("run_experiments.py", "results.json", "experiments.stdout"),
                 ("run_shrink_control.py", "results_shrink_control.json", "shrink.log"),
                 ("run_diagnostics.py", "diagnostics.json", "diagnostics.log"),
                 ("run_retrieval.py", "results_retrieval.json", "retrieval.stdout"),
                 ("run_time_local.py", "results_time_local.json", "time_local.stdout"))
        for script, out, log in steps:
            if not os.path.exists(f"{W}/{out}"):
                run(script, W, log, **env)
        r = json.load(open(f"{W}/encoder_log.json"))["res"]["test"]
        print(f"    энкодер на тесте NDCG@10 {r['ndcg10']:.4f}, популярность {r['pop_ndcg10']:.4f}")

пропускаю churn, нет данных


# Сводка и сохранение

Скопируй весь вывод следующей ячейки и пришли мне вместе с архивом из последней ячейки.

In [8]:
for s in SPLITS:
    W = f"{WORK}/split{s}"
    if os.path.exists(f"{W}/results.json"):
        print("=" * 30, f"SASRec, разбиение {s}", "=" * 30)
        show("summarize.py", W)
done = [f"{WORK}/split{s}" for s in SPLITS if os.path.exists(f"{WORK}/split{s}/results.json")]
if done:
    print("=" * 30, "SASRec, все разбиения вместе", "=" * 30)
    show("combine_splits.py", WORK, *done)
for n in BANK:
    banks = [f"{WORK}/bank_{n}_split{s}" for s in BANK_SPLITS if os.path.exists(f"{WORK}/bank_{n}_split{s}/results_time_local.json")]
    for W in banks:
        print("=" * 30, os.path.basename(W), "=" * 30)
        print(json.load(open(f"{W}/encoder_log.json"))["info"])
        show("summarize.py", W)
    if len(banks) > 1:
        show("combine_splits.py", WORK, *banks)

============================== SASRec, разбиение 0 ==============================
Результаты из папки /content/drive/MyDrive/ext_work_v5/split0

SASRec авторов на чистом протоколе
  пользователи {'train': 4827, 'val': 402, 'test': 805}, товаров 3125, взаимодействий 574376
  предобученный SASRec, NDCG@10 на val 0.1207, на test 0.1099
  лишние эпохи без контекста, эпоха по val 3, test 0.1080, к предобученному -0.0018 [-0.0076; +0.0038]
  если бы эпоху выбирали по тесту, было бы 0.1115 на эпохе 4
  агрегация авторов learnable_attention|freeze=0|alpha=0.1
    test при эпохе по val 0.1053, при выборе по тесту было бы 0.1113
    к предобученному -0.0046 [-0.0116; +0.0022]
    к тем же эпохам без контекста -0.0027 [-0.0092; +0.0033]
    в исходном коде авторов контекст из будущего получили бы 0.28% обучающих позиций и 0.12% тестовых запросов
  агрегация авторов mean|freeze=0|alpha=0.1
    test при эпохе по val 0.1081, при выборе по тесту было бы 0.1128
    к предобученному -0.0018 [-0.0078; +

In [9]:
!cd {WORK} && rm -f /content/results_v5.zip && zip -q -r /content/results_v5.zip . -x "*.npz" "*.pkl" "*.pth" "*.zip" "*.parquet"
if USE_DRIVE:
    shutil.copy("/content/results_v5.zip", WORK)
from google.colab import files
files.download("/content/results_v5.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Как читать результаты

В части A главное сравнение это агрегация авторов против тех же 20 эпох без контекста. Если на всех разбиениях прирост около нуля, то прирост в статье объясняется лишними эпохами и выбором эпохи по тесту.

В части B смотри на блок время против похожести. Если текущая популярность (случайные переходы за последние дни) обгоняет популярность за всё время, значит в данных есть общий для всех сдвиг, и именно его ловит среднее поле статьи. Если похожие переходы за последние дни дают что-то поверх всей памяти (строка про смесь всей и свежей памяти), значит работает соединение двух идей, похожей ситуации и совпадения по времени. Это и есть кандидат в метод, который продолжает статью, а не заменяет её. Критерий нестационарности должен с этим согласовываться. Отношение доли остатка, объяснённой неделей, к той же доле при перемешанных неделях заметно больше единицы как раз там, где контекст в целом помогает.

Если что-то пошло не так. Ошибка ModuleNotFoundError про src.data значит, что не выполнилась ячейка A1. При нехватке памяти CUDA в шаге A2 добавь к команде --eval_batch 16. Если упал шаг numpy-конвейера, ячейка покажет хвост его лога, его и пришли.